# 1. Header, Citations, Purpose & Topic Bridges

**Topic:** 1-Lipschitz Networks: Björck Orthonormalization, Cayley Transform, and GroupSort
**Category:** 06_smooth_and_physics_informed_ai

### Citations & Core References
- Björck, Å., & Bowie, C. (1971). *An Iterative Algorithm for Computing the Best Estimate of an Orthogonal Matrix*. SIAM Journal on Numerical Analysis.
- Anil, C., Lucas, J., & Grosse, R. (2019). *Sorting Out Lipschitz Function Approximation*. ICML.
- Trotter, H. F. (1959). *On the Product of Semi-Groups of Operators*. Proceedings of the AMS.
- Huster, T., et al. (2018). *Limitations of the Lipschitz Constant of Self-Attention*. arXiv:1808.01665.

## 2. Mathematical Foundations & Governing Equations

### Björck Orthonormalization Iteration
$$W_0 = \frac{W}{\|W\|_F}, \quad W_{k+1} = W_k (1.5 I - 0.5 W_k^T W_k)$$

### Cayley Orthogonal Parameterization
$$A = M - M^T, \quad Q = (I - A)(I + A)^{-1}, \quad Q^T Q = I$$

### GroupSort-2 Activation (PairSort)
$$\text{GroupSort}_2(x_{2i}, x_{2i+1}) = \left(\max(x_{2i}, x_{2i+1}), \min(x_{2i}, x_{2i+1})\right)$$

In [ ]:
# 3. Deterministic Setup & Micro-Data Generator
import torch

torch.manual_seed(42)

# Weight setup for Björck: D=6
D = 6
w_raw = torch.randn(D, D)

# Cayley setup: D_c=4
D_c = 4
m_raw = torch.randn(D_c, D_c) * 0.5

# Input features: B=3, D=6
B = 3
x_feat = torch.randn(B, D)
print(f"Setup complete: w_raw {w_raw.shape}, m_raw {m_raw.shape}, x_feat {x_feat.shape}")

In [ ]:
# 4. First-Principles Implementation & Native Parity
def bjorck_orthonormalize(w, iters=5):
    norm = torch.linalg.matrix_norm(w, ord="fro") + 1e-6
    w_k = w / norm
    d_out = w.shape[1]
    eye = torch.eye(d_out, dtype=w.dtype, device=w.device)
    for _ in range(iters):
        gram = w_k.t() @ w_k
        w_k = w_k @ (1.5 * eye - 0.5 * gram)
    return w_k

def cayley_transform(m):
    d = m.shape[0]
    eye = torch.eye(d, dtype=m.dtype, device=m.device)
    a = m - m.t()
    return (eye - a) @ torch.linalg.inv(eye + a)

def groupsort2(x):
    b, d = x.shape
    pairs = x.view(b, d // 2, 2)
    sorted_pairs, _ = torch.sort(pairs, dim=-1, descending=True)
    return sorted_pairs.reshape(b, d)

w_ortho = bjorck_orthonormalize(w_raw, iters=5)
q_cayley = cayley_transform(m_raw)
h_bjorck = x_feat @ w_ortho
y_groupsort = groupsort2(h_bjorck)
print(f"Ortho weight: {w_ortho.shape}, Cayley: {q_cayley.shape}, Output: {y_groupsort.shape}")

In [ ]:
# 5. Quantitative Invariants & Parity Verification
# Invariant 1: Björck orthonormalization produces approximate isometry (W^T W ~ I)
gram_bjorck = w_ortho.t() @ w_ortho
eye_bjorck = torch.eye(D)
diff_bjorck = torch.max(torch.abs(gram_bjorck - eye_bjorck)).item()
assert diff_bjorck < 1e-3, f"Björck orthogonality residual too high: {diff_bjorck}"

# Invariant 2: Cayley transform produces exact isometry (Q^T Q = I)
gram_cayley = q_cayley.t() @ q_cayley
eye_cayley = torch.eye(D_c)
diff_cayley = torch.max(torch.abs(gram_cayley - eye_cayley)).item()
assert diff_cayley < 1e-5, f"Cayley orthogonality failed: {diff_cayley}"

# Invariant 3: GroupSort-2 preserves L2 norm exactly (isometry)
norm_in = torch.norm(h_bjorck, p=2, dim=-1)
norm_out = torch.norm(y_groupsort, p=2, dim=-1)
assert torch.allclose(norm_in, norm_out, atol=1e-5), "GroupSort violated L2 norm conservation"
print("All mathematical invariants (Stiefel manifold projection & isometry) verified.")

In [ ]:
# 6. Performance Profile & Diagnostic Visualizations
import time

iters = 100
t0 = time.perf_counter()
for _ in range(iters):
    _ = bjorck_orthonormalize(w_raw, iters=5)
    _ = cayley_transform(m_raw)
    _ = groupsort2(h_bjorck)
dt = (time.perf_counter() - t0) / iters * 1000.0
print(f"Fused 1-Lipschitz step latency: {dt:.3f} ms / step")

In [ ]:
# 7. Pathological Misuse vs. Corrected Implementation
# Pathological: unscaled Björck iteration diverges exponentially when ||W||_2 > sqrt(3)
w_huge = w_raw * 10.0
w_divergent = w_huge.clone()
eye = torch.eye(D)
for _ in range(5):
    gram = w_divergent.t() @ w_divergent
    w_divergent = w_divergent @ (1.5 * eye - 0.5 * gram)
is_exploded = torch.isnan(w_divergent).any() or torch.isinf(w_divergent).any() or w_divergent.abs().max() > 1e10
print(f"Pathological unscaled Björck exploded: {is_exploded}")

# Corrected: Frobenius scaling guarantees convergence within the polynomial basin
w_stable = bjorck_orthonormalize(w_huge, iters=5)
print(f"Corrected Björck max singular value diff: {torch.abs(torch.linalg.matrix_norm(w_stable, 2) - 1.0).item():.4f}")

## 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

1. **Robustness Without Adversarial Training**: 1-Lipschitz networks guarantee certified robustness against norm-bounded perturbations by construction.
2. **Expressive Activations**: GroupSort avoids the gradient-diminishing contractive flaw of ReLU, preserving gradient norm through the entire depth.
3. **Universal Isometry**: Orthonormal linear maps combined with GroupSort form universal approximators for any 1-Lipschitz continuous function.